# Generación GTFS: trips


In [1]:
import pandas as pd
from pathlib import Path
import geopandas as gpd

## Parámetros

In [2]:
import json
from pathlib import Path as PathLib

_params_path = PathLib.cwd() / "params.json"
if not _params_path.exists():
    _params_path = PathLib("params.json")
with open(_params_path, encoding="utf-8") as f:
    p = json.load(f)

CIUDAD = p["ciudad"]
AGENCY_NAME = p["agency"]["name"]
AGENCY_ID = p["agency"]["id"]
AGENCY_URL = p["agency"]["url"]
AGENCY_TIMEZONE = p["agency"]["timezone"]
AGENCY_LANG = p["agency"]["lang"]
START_DATE = p["calendar"]["start_date"]
END_DATE = p["calendar"]["end_date"]
SERVICE_ID_VALLE = p["calendar"]["service_id_valle"]
SERVICE_ID_PICO = p["calendar"]["service_id_pico"]
SERVICE_ID_FINDE = p["calendar"]["service_id_finde"]
SERVICE_ID = SERVICE_ID_VALLE

In [3]:
# Parámetros de calendar y service_id cargados desde params.json en la celda anterior

In [4]:
# --- Carpeta GTFS ---
PATH_DIR_GTFS = Path(f"../data/{CIUDAD}/gtfs-output")
PATH_DIR_GTFS.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_GTFS.absolute()}")

# --- Carpeta proccesed ---
PATH_DIR_proccesed = Path(f"../data/{CIUDAD}/processed")
PATH_DIR_proccesed.mkdir(parents=True, exist_ok=True)
print(f"Salida: {PATH_DIR_proccesed.absolute()}")

Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../data/tampico/gtfs-output
Salida: /Users/danielbustillos/Documents/ITDP/proyectos-ITDP/generation-gtfs-from-shapes-v2/2-generation-files-simples/../data/tampico/processed


## Read files

In [5]:
stop_times_df = pd.read_csv(PATH_DIR_GTFS / "stop_times.txt")
stop_times_df.head()

,trip_id,timepoint,stop_id,stop_sequence,arrival_time,departure_time
0,Route_1_trip_00,1,Route_1_0000,1,00:00:00,00:00:12
1,Route_1_trip_00,1,Route_1_0001,2,00:00:37,00:00:49
2,Route_1_trip_00,1,Route_1_0002,3,00:01:15,00:01:27
3,Route_1_trip_00,1,Route_1_0003,4,00:01:52,00:02:04
4,Route_1_trip_00,1,Route_1_0004,5,00:02:30,00:02:42


In [6]:
routes_gdf = gpd.read_file(PATH_DIR_proccesed / "routes_clean.geojson")
routes_gdf

,route_id,agency_id,route_short_name,route_long_name,route_type,shape_id,geometry
0,Route_1,IMEPLAN_Tampico,Route_1,Mirador - Aviación por Boulevard,3,Shape_Route_1,"LINESTRING (618098.532 2457141.016, 617928.009..."
1,Route_2,IMEPLAN_Tampico,Route_2,Tampico - Bosque Central de Abastos - Casa Blanca,3,Shape_Route_2,"LINESTRING (614915.384 2470530.991, 614995.778..."
2,Route_3,IMEPLAN_Tampico,Route_3,Madero - Revolución Verde - Central de Abastos,3,Shape_Route_3,"LINESTRING (614921.605 2470525.501, 615001.925..."
3,Route_9,IMEPLAN_Tampico,Route_9,Santa Elena - Colonias - Tampico,3,Shape_Route_9,"LINESTRING (618703.105 2469648.325, 618782.266..."
4,Route_4,IMEPLAN_Tampico,Route_4,Colosio - Águila Madero - Golfo,3,Shape_Route_4,"LINESTRING (613305.943 2469210.556, 613305.523..."
...,...,...,...,...,...,...,...
133,Route_118,IMEPLAN_Tampico,Route_118,Frac. Haciendas - Soriana - Aeropuerto,3,Shape_Route_118,"LINESTRING (616065.752 2474121.092, 615961.842..."
134,Route_117,IMEPLAN_Tampico,Route_117,Altamira - San Jacinto - Paseo del Real,3,Shape_Route_117,"LINESTRING (609541.207 2477226.635, 609507.577..."
135,Route_119,IMEPLAN_Tampico,Route_119,Pedrera - Soriana - Aeropuerto,3,Shape_Route_119,"LINESTRING (615436.96 2475730.583, 615465.083 ..."
136,Route_FRACC,IMEPLAN_Tampico,Route_FRACC,Sotavento - Soriana Aeropuerto - AUT,3,Shape_Route_FRACC,"LINESTRING (614477.977 2470906.392, 614512.061..."


## Construcción de trips.txt

A partir de los viajes únicos en `stop_times_df` (trip_id, route_id), se asigna un único `service_id`, se incorpora `shape_id` desde `routes_gdf` y se arma la tabla GTFS: route_id, service_id, trip_id, trip_headsign, direction_id, shape_id.

In [7]:
stop_times_df["route_id"] = "Route_" + stop_times_df["stop_id"].str.split("_").str[1]
stop_times_df

,trip_id,timepoint,stop_id,stop_sequence,arrival_time,departure_time,route_id
0,Route_1_trip_00,1,Route_1_0000,1,00:00:00,00:00:12,Route_1
1,Route_1_trip_00,1,Route_1_0001,2,00:00:37,00:00:49,Route_1
2,Route_1_trip_00,1,Route_1_0002,3,00:01:15,00:01:27,Route_1
3,Route_1_trip_00,1,Route_1_0003,4,00:01:52,00:02:04,Route_1
4,Route_1_trip_00,1,Route_1_0004,5,00:02:30,00:02:42,Route_1
...,...,...,...,...,...,...,...
23277,Route_FRACC_trip_00,1,Route_FRACC_0129,130,01:20:49,01:21:01,Route_FRACC
23278,Route_FRACC_trip_00,1,Route_FRACC_0130,131,01:21:27,01:21:39,Route_FRACC
23279,Route_FRACC_trip_00,1,Route_FRACC_0131,132,01:22:05,01:22:17,Route_FRACC
23280,Route_FRACC_trip_00,1,Route_FRACC_0132,133,01:22:42,01:22:54,Route_FRACC


In [8]:
# Viajes únicos: una fila por (trip_id, route_id)
trips_base = (
    stop_times_df[["trip_id", "route_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# route_id debe coincidir con routes.txt (p. ej. Route_11_A, Route_11_B, no Route_11)
# Se deriva del trip_id: "Route_11_A_trip_00" -> "Route_11_A"
trips_base["route_id"] = trips_base["trip_id"].str.replace("_trip_00$", "", regex=True)

# shape_id desde routes_gdf (route_id → shape_id)
routes_shape = routes_gdf[["route_id", "shape_id"]].drop_duplicates()
trips_base = trips_base.merge(routes_shape, on="route_id", how="left")

# Asignar service_id único y columnas opcionales GTFS
trips_base["service_id"] = SERVICE_ID
trips_base["trip_headsign"] = ""
trips_base["direction_id"] = 1

# Orden de columnas para trips.txt
cols_trips = ["route_id", "service_id", "trip_id", "trip_headsign", "direction_id", "shape_id"]
trips_df = trips_base[cols_trips]

trips_df.head(10)

,route_id,service_id,trip_id,trip_headsign,direction_id,shape_id
0,Route_1,L_V_VALLE,Route_1_trip_00,,1,Shape_Route_1
1,Route_10,L_V_VALLE,Route_10_trip_00,,1,Shape_Route_10
2,Route_100,L_V_VALLE,Route_100_trip_00,,1,Shape_Route_100
3,Route_100A,L_V_VALLE,Route_100A_trip_00,,1,Shape_Route_100A
4,Route_101,L_V_VALLE,Route_101_trip_00,,1,Shape_Route_101
5,Route_101A,L_V_VALLE,Route_101A_trip_00,,1,Shape_Route_101A
6,Route_102,L_V_VALLE,Route_102_trip_00,,1,Shape_Route_102
7,Route_103,L_V_VALLE,Route_103_trip_00,,1,Shape_Route_103
8,Route_103A,L_V_VALLE,Route_103A_trip_00,,1,Shape_Route_103A
9,Route_103B,L_V_VALLE,Route_103B_trip_00,,1,Shape_Route_103B


## Export

In [9]:
trips_df.to_csv(PATH_DIR_GTFS / "trips.txt", index=False)
print(f"Escrito: {PATH_DIR_GTFS / 'trips.txt'} ({len(trips_df)} viajes)")

Escrito: ../data/tampico/gtfs-output/trips.txt (138 viajes)
